## بصيرة | Baseera

دفتر تدريب واختبار نموذج اكتشاف جدول القيم الغذائية باستخدام YOLOv8s.

**تسلسل التدريب المعتمد:**
1. Baseline: 25 Epochs على النسخة الأولى من البيانات.
2. `baseera_v2_run3`: 15 Epochs على النسخة الثانية (226 صورة)، بدءًا من أفضل أوزان Baseline.
3. `baseera_v2_run4`: 20 Epochs إضافية على النسخة الثانية، بدءًا من `last.pt` للمرحلة الثانية.

**قبل التشغيل:** اختاري GPU من إعدادات Google Colab، وتأكدي من توفر الإصدارين الأول والثاني من مجموعة البيانات في Roboflow. أدخلي مفتاح Roboflow عند الطلب؛ لا تضعيه داخل الكود أو مخرجات الدفتر.

**النموذج النهائي للتقييم والتنبؤ:** `baseera_v2_run4/weights/best.pt`.

---

### طريقة تشغيل النتائج المعتمدة

النتائج الواردة في هذا الدفتر مأخوذة من تشغيل **الأوزان النهائية الجاهزة** (`best.pt`) المحمّلة من المستودع في خلية «تحميل الأوزان»، **دون إعادة تنفيذ التدريب**. خلايا التدريب (المراحل الأولى والثانية والثالثة) توثّق خطوات التدريب السابقة، وتشغيلها يعيد التدريب ويكتب فوق الأوزان المحمّلة، وقد تختلف النتائج قليلًا. خلية حفظ الأوزان في Google Drive تعتمد على نتائج التدريب، فتُتخطّى أيضًا عند استخدام الأوزان الجاهزة.

**للتحقق من النتائج:** شغّلي خلايا الإعداد وتحميل البيانات والأوزان، ثم خلايا التقييم والتنبؤ (من قسم «التحقق باستخدام Ultralytics» فصاعدًا)، وتخطّي خلايا التدريب. شغّلي خلية تقييم Test قبل خلية عرض مصفوفة الالتباس.

### النتائج المعتمدة

| المجموعة | Precision | Recall | mAP50 | mAP50-95 |
|---|---|---|---|---|
| Validation | 0.7266 | 0.7143 | 0.6966 | 0.4087 |
| Test | 0.5656 | 1.0000 | 0.8616 | 0.4696 |

- **التنبؤ (عتبة الثقة 0.25):** 10 صور اختبار، ظهر اكتشاف في 9 منها، بإجمالي 21 اكتشافًا.
- **Confusion Matrix (Test):** `[[8, 12], [0, 0]]`.

### قيود

- مجموعة الاختبار صغيرة (10 صور و8 عناصر حقيقية)، فالنتائج غير كافية للتعميم.
- النموذج يكتشف **جدول القيم الغذائية فقط**، ولا يستخرج السكر أو حجم الحصة ولا ينفّذ OCR.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Pip install method (recommended)

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
from getpass import getpass

# أدخلي المفتاح عند التشغيل؛ لا تكتبيه داخل الدفتر.
rf = Roboflow(api_key=getpass("Roboflow API Key: "))
project = rf.workspace("halahs-workspace").project("baseera")

# النسخة الأولى: بيانات المرحلة الأساسية.
baseline_version = project.version(1)
baseline_dataset = baseline_version.download("yolov8")

# النسخة الثانية (226 صورة): تُستخدم في مرحلتي الاستكمال.
version2 = project.version(2)
dataset = version2.download("yolov8")

print("تم تحميل النسختين: baseline ونسخة التدريب الثانية.")

In [ ]:

import os
import urllib.request

# رابط النموذج على GitHub
weights_url = "https://raw.githubusercontent.com/FaHusa2003/Baseera/main/best.pt"

# المسار الذي تتوقعه خلايا التقييم والتنبؤ
final_model_path = f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt"

# إنشاء المجلد إذا لم يكن موجودًا
os.makedirs(os.path.dirname(final_model_path), exist_ok=True)

# تحميل النموذج
print("جاري تحميل أوزان نموذج بصيرة من GitHub...")

urllib.request.urlretrieve(weights_url, final_model_path)

# التحقق من نجاح التحميل
if os.path.isfile(final_model_path) and os.path.getsize(final_model_path) > 1_000_000:
    print("تم تحميل النموذج بنجاح.")
    print(f"مسار النموذج: {final_model_path}")
    print(f"حجم الملف: {os.path.getsize(final_model_path) / (1024 * 1024):.2f} MB")
else:
    raise RuntimeError("فشل تحميل ملف النموذج أو أن الملف غير مكتمل.")


## المرحلة الأولى: التدريب الأساسي (Baseline)

تدريب أولي لمدة 25 دورة على النسخة الأولى من البيانات. تُستخدم أوزان هذه المرحلة نقطةَ انطلاق للمرحلة التالية.

**إعدادات المرحلة:** YOLOv8s، ومقاس صور 512×512، و25 Epochs.

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={baseline_dataset.location}/data.yaml epochs=25 imgsz=512 plots=True project={HOME}/runs/detect name=baseera_baseline exist_ok=True

## المرحلة الثانية: baseera_v2_run3

تُستخدم مجموعة البيانات الثانية (226 صورة)، ويبدأ التدريب من أفضل أوزان المرحلة الأساسية `baseera_baseline/weights/best.pt`.

**إعدادات المرحلة:** 15 Epochs، ومقاس صور 512×512، وBatch 4.

In [ ]:
import numpy as np
from ultralytics import YOLO
import os

# توافق دالة trapz مع إصدارات NumPy الحديثة.
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

baseline_best = f"{HOME}/runs/detect/baseera_baseline/weights/best.pt"
if not os.path.isfile(baseline_best):
    raise FileNotFoundError(
        f"لم يُعثر على أوزان المرحلة الأساسية: {baseline_best}. "
        "شغّلي خلية التدريب الأساسي أولًا."
    )

model = YOLO(baseline_best)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=15,
    imgsz=512,
    batch=4,
    project=f"{HOME}/runs/detect",
    name="baseera_v2_run3",
    exist_ok=True,
)

## المرحلة الثالثة: baseera_v2_run4

استكمال التدريب لمدة 20 دورة إضافية على النسخة الثانية من البيانات، بدءًا من `last.pt` الناتج عن المرحلة الثانية `baseera_v2_run3`.

وبذلك يصبح مجموع دورات التدريب على النسخة الثانية 35 دورة (15 + 20).

In [ ]:
import numpy as np
from ultralytics import YOLO
import os

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

stage2_last = f"{HOME}/runs/detect/baseera_v2_run3/weights/last.pt"
if not os.path.isfile(stage2_last):
    raise FileNotFoundError(
        f"لم يُعثر على أوزان المرحلة الثانية: {stage2_last}. "
        "شغّلي خلية baseera_v2_run3 أولًا."
    )

model = YOLO(stage2_last)

results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=20,
    imgsz=512,
    batch=4,
    project=f"{HOME}/runs/detect",
    name="baseera_v2_run4",
    exist_ok=True,
)

In [ ]:
import os, shutil
from google.colab import drive

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/Baseera_AI_Weights"
os.makedirs(save_dir, exist_ok=True)

src_run4 = f"{HOME}/runs/detect/baseera_v2_run4"
final_best = f"{src_run4}/weights/best.pt"

if not os.path.isfile(final_best):
    raise FileNotFoundError(
        f"لم يُعثر على أفضل أوزان المرحلة النهائية: {final_best}. "
        "تأكدي من نجاح تدريب baseera_v2_run4."
    )

shutil.copytree(src_run4, f"{save_dir}/baseera_v2_run4", dirs_exist_ok=True)
shutil.copy2(final_best, f"{save_dir}/baseera_v2_best.pt")

print("تم حفظ أوزان المرحلة النهائية baseera_v2_run4 في Google Drive.")

In [ ]:
import glob, os, shutil

# إنشاء ملف test_images.zip من صور مجموعة الاختبار (حتى 10 صور).
os.makedirs("/content/test_images", exist_ok=True)

# 2. البحث عن صور داخل أي مجلد test/images
imgs = sorted(
    glob.glob("/content/**/test/images/*.jpg", recursive=True)
    + glob.glob("/content/**/test/images/*.png", recursive=True)
    + glob.glob("/content/**/test/images/*.jpeg", recursive=True)
)[:10]

# 3. نسخ الصور إلى مجلد test_images
for p in imgs:
  shutil.copy2(p, "/content/test_images/")

print("عدد الصور المنسوخة:", len(os.listdir("/content/test_images")))

# 4. ضغط المجلد بالكامل إلى test_images.zip
shutil.make_archive("/content/test_images", "zip", "/content/test_images")
print("تم إنشاء الملف المضغوط: /content/test_images.zip")

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

final_model_path = f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt"
if not os.path.isfile(final_model_path):
    raise FileNotFoundError("شغّلي مراحل التدريب حتى baseera_v2_run4 قبل التقييم.")

model = YOLO(final_model_path)

# قراءة أسماء الفئات من data.yaml أو من النموذج
class_names = list(model.names.values()) if isinstance(model.names, dict) else list(model.names)
num_classes = len(class_names)

# مجلد صور وتسميات التحقق (valid)
val_images = sorted(glob.glob(f"{dataset.location}/valid/images/*"))
label_dir = os.path.join(dataset.location, "valid", "labels")

def iou_box(box1, box2):
    # box: [x1, y1, x2, y2]
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0.0

# مصفوفة الالتباس: الصفوف = الحقيقي (True) + background، الأعمدة = المتوقع (Pred) + background
cm = np.zeros((num_classes + 1, num_classes + 1), dtype=int)

conf_thresholds = np.linspace(0.05, 0.95, 19)
f1_scores = []

for conf_th in conf_thresholds:
    tp, fp, fn = 0, 0, 0
    for img_path in val_images:
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(label_dir, stem + ".txt")

        res = model.predict(img_path, imgsz=800, conf=float(conf_th), verbose=False)[0]
        h, w = res.orig_shape

        # قراءة الصناديق الحقيقية بصيغة YOLO (cls cx cy bw bh)
        gt_boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        c = int(float(parts[0]))
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = (cx - bw / 2) * w
                        y1 = (cy - bh / 2) * h
                        x2 = (cx + bw / 2) * w
                        y2 = (cy + bh / 2) * h
                        gt_boxes.append([c, x1, y1, x2, y2, False])

        pred_boxes = []
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            cls_ids = res.boxes.cls.cpu().numpy().astype(int)
            confs = res.boxes.conf.cpu().numpy()
            for i in range(len(cls_ids)):
                pred_boxes.append([cls_ids[i], xyxy[i][0], xyxy[i][1], xyxy[i][2], xyxy[i][3], confs[i], False])

        # مطابقة التوقعات مع الحقيقي عند IoU >= 0.5
        for p in pred_boxes:
            best_iou = 0.0
            best_idx = -1
            for idx, g in enumerate(gt_boxes):
                if not g[5]:
                    iou = iou_box(p[1:5], g[1:5])
                    if iou > best_iou:
                        best_iou = iou
                        best_idx = idx
            if best_idx >= 0 and best_iou >= 0.5:
                gt_boxes[best_idx][5] = True
                p[6] = True
                if p[0] == gt_boxes[best_idx][0]:
                    tp += 1
                else:
                    fp += 1
                if abs(conf_th - 0.25) < 0.03:
                    cm[gt_boxes[best_idx][0], p[0]] += 1
            else:
                fp += 1
                if abs(conf_th - 0.25) < 0.03:
                    cm[num_classes, p[0]] += 1

        for g in gt_boxes:
            if not g[5]:
                fn += 1
                if abs(conf_th - 0.25) < 0.03:
                    cm[g[0], num_classes] += 1

    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    f1_scores.append(f1)

# 1) رسم منحنى الثقة مع F1
plt.figure(figsize=(6.5, 4.2))
plt.plot(conf_thresholds, f1_scores, marker="o", linewidth=2)
plt.title("Confidence vs F1 Curve")
plt.xlabel("Confidence Threshold")
plt.ylabel("F1 Score")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2) رسم مصفوفة الالتباس
labels_all = class_names + ["background"]
plt.figure(figsize=(6.5, 5.2))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix (conf ~ 0.25)")
plt.colorbar()
ticks = np.arange(len(labels_all))
plt.xticks(ticks, labels_all, rotation=35, ha="right")
plt.yticks(ticks, labels_all)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center",
                 color="white" if cm[i, j] > (cm.max() / 2 if cm.max() > 0 else 1) else "black")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## التحقق باستخدام Ultralytics

تشغيل التقييم القياسي على مجموعة **Validation** باستخدام أفضل أوزان المرحلة النهائية `baseera_v2_run4/weights/best.pt`.

In [ ]:
import os
import numpy as np

os.chdir(HOME)

if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

from ultralytics import YOLO

model = YOLO(f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt")
metrics = model.val(data=f"{dataset.location}/data.yaml", split="val")

In [ ]:

import numpy as np

# معالجة التوافق بين NumPy وUltralytics
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

from ultralytics import YOLO

# تحميل النموذج النهائي
model = YOLO(
    f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt"
)

# تنفيذ التحقق على مجموعة Validation
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="val"
)

# عرض المقاييس
print("\nنتائج التحقق:")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")


## التنبؤ على مجموعة Test

تشغيل النموذج النهائي على صور مجموعة **Test**. هذه الخطوة تعرض التنبؤات ولا تُعدّ وحدها حسابًا لمقاييس الاختبار النهائية.

In [ ]:
%cd {HOME}

!yolo task=detect mode=predict model={HOME}/runs/detect/baseera_v2_run4/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

### عرض بعض نتائج التنبؤ


In [ ]:
import base64
import glob
import os
from IPython.display import HTML, display

base_path = f"{HOME}/runs/detect/"
subfolders = [
    os.path.join(base_path, d)
    for d in os.listdir(base_path)
    if os.path.isdir(os.path.join(base_path, d)) and d.startswith("predict")
]

if subfolders:
    latest_folder = max(subfolders, key=os.path.getmtime)
    image_paths = sorted(glob.glob(f"{latest_folder}/*.jpg"))

    if image_paths:
        print(f"عدد الصور: {len(image_paths)}")
        imgs_html = []
        for p in image_paths:
            with open(p, "rb") as f:
                b64 = base64.b64encode(f.read()).decode("utf-8")
            name = os.path.basename(p)
            imgs_html.append(
                f'<div style="text-align:center; max-width:320px;">'
                f'<img src="data:image/jpeg;base64,{b64}" style="width:100%; height:auto; border-radius:6px;" />'
                f'<div style="font-size:12px; margin-top:4px;">{name}</div>'
                f"</div>"
            )

        display(
            HTML(
                f'<div style="display:flex; flex-wrap:wrap; gap:12px; align-items:flex-start;">'
                f'{"".join(imgs_html)}'
                f"</div>"
            )
        )
    else:
        print("لم يتم العثور على صور داخل مجلد التنبؤ.")
else:
    print("لم يتم العثور على مجلد نتائج التنبؤ.")

In [ ]:

import numpy as np

# معالجة توافق NumPy
if not hasattr(np, "trapz"):
    np.trapz = np.trapezoid

from ultralytics import YOLO

# تحميل النموذج النهائي
model = YOLO(
    f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt"
)

# تقييم النموذج على مجموعة Test
test_metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

# عرض النتائج النهائية
print("\n========== Test Results ==========")
print(f"Precision: {test_metrics.box.mp:.4f}")
print(f"Recall: {test_metrics.box.mr:.4f}")
print(f"mAP50: {test_metrics.box.map50:.4f}")
print(f"mAP50-95: {test_metrics.box.map:.4f}")


In [ ]:
from IPython.display import Image, display
import os

# مجلد نتائج تقييم Test من خلية التقييم السابقة مباشرة (شغّلي تلك الخلية أولًا)
results_dir = str(test_metrics.save_dir)
print("مجلد النتائج:", results_dir)

for filename in ["confusion_matrix.png", "PR_curve.png"]:
    path = os.path.join(results_dir, filename)
    if os.path.exists(path):
        print(filename)
        display(Image(filename=path, width=650))
    else:
        print(f"لم يُعثر على {filename}")


In [ ]:

from ultralytics import YOLO
import numpy as np

model = YOLO(
    f"{HOME}/runs/detect/baseera_v2_run4/weights/best.pt"
)

results = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test",
    plots=True
)

print("\nConfusion Matrix:")
print(results.confusion_matrix.matrix)

print("\nClass Names:")
print(results.names)


In [ ]:
print(dataset.location)

with open(f"{dataset.location}/data.yaml", "r") as f:
    print(f.read())